# Notebook 3.2: Passive Tracer Transport (Convection-Diffusion)

## Objective
Solve the unsteady convection-diffusion equation for a passive scalar in Poiseuille flow:
$$\frac{\partial c}{\partial t} + \mathbf{u} \cdot \nabla c = D \nabla^2 c$$

**Setup:**
- Velocity field: Poiseuille (from notebook 2)
- Tracer injected at inlet: $c=1$ at $x=0$, $c=0$ initially everywhere else
- Watch the concentration front evolve and spread

**Implicit Euler** time-stepping is used for unconditional stability.

In [ ]:
from fenics import *
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
set_log_level(LogLevel.WARNING)

---
## Step 1: Solve Poiseuille Flow

In [ ]:
L, H, mu, dp = 1.0, 0.1, 1.0, 1.0

mesh = RectangleMesh(Point(0.0, -H/2), Point(L, H/2), 60, 15)

# Mixed space (Taylor-Hood P2-P1)
W = FunctionSpace(mesh, MixedElement([
    VectorElement("P", mesh.ufl_cell(), 2),
    FiniteElement("P", mesh.ufl_cell(), 1)
]))

tol = 1e-10
noslip = Constant((0.0, 0.0))
bcs = [
    DirichletBC(W.sub(0), noslip, f"on_boundary && (x[1] > {H/2 - tol} || x[1] < {-H/2 + tol})"),
]

(u, p) = TrialFunctions(W)
(v, q) = TestFunctions(W)

a_s = (mu * inner(grad(u), grad(v)) - p*div(v) + q*div(u)) * dx
L_s = dot(Constant((-dp/L, 0.0)), v) * dx

w = Function(W)
solve(a_s == L_s, w, bcs)
u_flow, _ = w.split()

U_max = u_flow.vector().norm('linf')
print(f"Poiseuille solved. Max velocity: {U_max:.5f}  (analytical: {dp*H**2/(8*mu*L):.5f})")

---
## Step 2: Setup Transport Problem

Implicit Euler weak form:
$$\int \frac{c^{n+1} - c^n}{\Delta t} v \, dx + \int (\mathbf{u} \cdot \nabla c^{n+1}) v \, dx + D \int \nabla c^{n+1} \cdot \nabla v \, dx = 0$$

In [ ]:
D   = 0.002  # Diffusivity
dt  = 0.05   # Time step
T   = 2.0    # End time

Pe = U_max * L / D
print(f"Péclet number Pe = {Pe:.1f}  ({'advection' if Pe > 1 else 'diffusion'} dominated)")

S = FunctionSpace(mesh, "P", 1)   # Scalar space for concentration

c   = TrialFunction(S)
phi = TestFunction(S)
c_n = Function(S)                  # Solution at previous time step

# Inlet BC: c = 1
bc_c = DirichletBC(S, Constant(1.0), f"on_boundary && x[0] < {tol}")

# Bilinear and linear forms (implicit Euler)
a_c = (c/dt * phi + dot(u_flow, grad(c)) * phi + D * dot(grad(c), grad(phi))) * dx
L_c = c_n/dt * phi * dx

print("Transport problem defined.")

---
## Step 3: Time Loop

In [ ]:
t = 0.0
snapshots = {}    # Store c at selected times
save_times = [0.25, 0.5, 1.0, 1.5, 2.0]

A_c = assemble(a_c)   # Assemble once (u_flow is fixed)
bc_c.apply(A_c)

c_sol = Function(S)

while t < T - 1e-8:
    t += dt
    b_c = assemble(L_c)
    bc_c.apply(b_c)
    solve(A_c, c_sol.vector(), b_c)
    c_n.assign(c_sol)

    for ts in save_times:
        if abs(t - ts) < dt/2 and ts not in snapshots:
            snapshots[ts] = c_sol.copy(deepcopy=True)
            print(f"  Saved t = {ts:.2f}")

print("Time loop complete.")

---
## Step 4: Visualise

In [ ]:
n_snap = len(snapshots)
fig, axes = plt.subplots(n_snap, 1, figsize=(13, 2.5*n_snap))
if n_snap == 1: axes = [axes]

for ax, (ts, c_snap) in zip(axes, sorted(snapshots.items())):
    im = plot(c_snap, ax=ax, cmap='RdYlBu_r', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='$c$')
    ax.set_title(f't = {ts:.2f}', fontsize=11)
    ax.set_ylabel('$y$')

axes[-1].set_xlabel('$x$')
plt.suptitle(f'Tracer concentration  (Pe = {Pe:.0f},  D = {D})', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/tracer_transport.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 5: Mixing Analysis

In [ ]:
# Profile at x = 0.8 at last saved time
c_final = list(sorted(snapshots.items()))[-1][1]
y_pts   = np.linspace(-H/2 + 1e-6, H/2 - 1e-6, 60)
c_prof  = np.array([c_final(0.8, y) for y in y_pts])

# Taylor dispersion: the parabolic profile smears the front
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(c_prof, y_pts, 'b-', lw=2)
ax.set_xlabel('$c$', fontsize=12)
ax.set_ylabel('$y$', fontsize=12)
ax.set_title('Concentration profile at $x = 0.8$', fontsize=12)
ax.set_xlim([-0.05, 1.05]); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/concentration_profile.png', dpi=150)
plt.show()

print(f"Profile: mean c = {c_prof.mean():.3f}, std = {c_prof.std():.3f}")
print(f"Note: parabolic flow causes Taylor dispersion (non-uniform front)")

---
## Summary

- **Implicit Euler** is unconditionally stable — time step limited only by accuracy, not stability.
- **Taylor dispersion:** The parabolic velocity profile stretches the concentration front. Faster fluid near the center carries tracer further downstream.
- **Péclet number:** Controls the balance. High Pe → sharp fronts; low Pe → smooth spreading.

**Exercise:** Set `D = 0.1` (low Pe). How does the mixing change? Try `D = 0.0001` (high Pe). What do you observe?